In [ ]:
from pathlib import Path
import os
print(Path.cwd())
print("models exists:", os.path.exists("models"))
print("configs exists:", os.path.exists("configs"))

In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys


def _mask_text(text: str, secrets: list[str] | None = None) -> str:
    if not secrets:
        return text
    masked = text
    for secret in secrets:
        if secret:
            masked = masked.replace(secret, "***")
    return masked


def _run(cmd, secrets: list[str] | None = None):
    printable_cmd = _mask_text(" ".join(cmd), secrets)
    print("$", printable_cmd)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        print(_mask_text(result.stdout, secrets))
    if result.returncode != 0:
        stderr_text = result.stderr.strip() or "(no stderr)"
        stderr_text = _mask_text(stderr_text, secrets)
        raise RuntimeError(f"Command failed ({result.returncode}): {printable_cmd}\n{stderr_text}")


def _is_project_root(path: Path) -> bool:
    return (path / "configs").exists() and (path / "models").exists()


def _find_project_root(start_path: Path) -> Path | None:
    candidates = [start_path, *start_path.parents]
    for candidate in candidates:
        if _is_project_root(candidate):
            return candidate
    return None


def _scan_content_for_project_root(content_dir: Path) -> Path | None:
    # Detect a workspace synced by Colab extension under an unknown directory name.
    for candidate in content_dir.rglob("*"):
        if candidate.is_dir() and _is_project_root(candidate):
            return candidate
    return None


def _build_clone_url(repo_url: str) -> tuple[str, list[str]]:
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token and repo_url.startswith("https://github.com/"):
        # Personal access token flow for private repositories in Colab.
        authenticated_url = repo_url.replace("https://", f"https://{token}@", 1)
        return authenticated_url, [token]
    return repo_url, []


def _resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    local_root = _find_project_root(cwd)
    if local_root is not None:
        return local_root

    content_dir = Path("/content")
    if content_dir.exists():
        known_candidates = [
            content_dir / "reid_investigation",
            content_dir / "drive" / "MyDrive" / "reid_investigation",
        ]
        for candidate in known_candidates:
            if _is_project_root(candidate):
                return candidate

        scanned_root = _scan_content_for_project_root(content_dir)
        if scanned_root is not None:
            return scanned_root

        default_repo_url = "https://github.com/ituvtu/research-reid.git"
        repo_url = os.environ.get("REID_REPO_URL", default_repo_url).strip()
        clone_url, clone_secrets = _build_clone_url(repo_url)
        clone_target = content_dir / "reid_investigation"
        if not clone_target.exists():
            print(f"Cloning repository for Colab: {repo_url};")
            try:
                _run(["git", "clone", clone_url, str(clone_target)], secrets=clone_secrets)
            except Exception as error:
                raise RuntimeError(
                    "Unable to clone repository to /content/reid_investigation; "
                    "for a private repository set %env GITHUB_TOKEN=<token> or use /content/drive/MyDrive/reid_investigation; "
                    f"details: {error}"
                ) from error

            if _is_project_root(clone_target):
                return clone_target

        if _is_project_root(clone_target):
            return clone_target

        raise RuntimeError(
            "Project source was not found in Colab (/content/reid_investigation or /content/drive/MyDrive/reid_investigation). "
            "If the repository is private, set %env GITHUB_TOKEN=<token> and optionally %env REID_REPO_URL=<url> before running Cell 2;"
        )

    raise RuntimeError(
        "Could not find a project root with configs and models directories; check the working directory;"
    )


PROJECT_ROOT = _resolve_project_root()
get_ipython().run_line_magic("cd", str(PROJECT_ROOT))

module_to_package = {
    "numpy": "numpy",
    "torch": "torch",
    "ultralytics": "ultralytics",
    "motmetrics": "motmetrics",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "yaml": "PyYAML",
}
required_modules = tuple(module_to_package.keys())
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]

print(f"Python executable: {sys.executable};")
print(f"Python version: {sys.version.split()[0]};")
print(f"Project root: {PROJECT_ROOT};")


def _detect_uv_cli() -> list[str] | None:
    uv_path = shutil.which("uv")
    if uv_path:
        return [uv_path]

    try:
        _run([sys.executable, "-m", "uv", "--version"])
        return [sys.executable, "-m", "uv"]
    except Exception:
        pass

    try:
        _run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
        _run([sys.executable, "-m", "pip", "install", "uv"])
        _run([sys.executable, "-m", "uv", "--version"])
        return [sys.executable, "-m", "uv"]
    except Exception:
        return None


def _install_packages(packages: list[str]) -> None:
    uv_cli = _detect_uv_cli()
    if uv_cli is not None:
        print("Installing dependencies via uv;")
        _run(uv_cli + ["pip", "install", "--python", sys.executable, *packages])
        return

    print("uv is unavailable - fallback to pip;")
    _run([sys.executable, "-m", "pip", "install", *packages])


if missing_modules:
    packages_to_install = [module_to_package[name] for name in missing_modules]
    print(f"Installing dependencies for active kernel: {', '.join(packages_to_install)};")
    _install_packages(packages_to_install)

failed_imports = []
for module_name in required_modules:
    try:
        importlib.import_module(module_name)
    except Exception:
        failed_imports.append(module_name)

if failed_imports:
    raise RuntimeError(
        "Failed to import modules after installation: "
        + ", ".join(failed_imports)
        + "; restart the kernel and run Cell 2 again;"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
if os.path.exists("/content") and "/content" not in sys.path:
    sys.path.append("/content")

print(f"Import path attached: {str(PROJECT_ROOT) in sys.path};")
print("Dependency status: ready;")

In [ ]:
from models.detectors import YOLODetector
from models.trackers import ByteTrackTracker
from utils import load_stage1_baseline_config, stage1_component_mappings

config = load_stage1_baseline_config('configs/stage1_baseline.yaml')
detector_config, tracker_config = stage1_component_mappings(config)

detector = YOLODetector.from_config(detector_config)
detector.load()
detector.warmup(image_size=(640, 640), runs=1)

tracker = ByteTrackTracker.from_config(tracker_config)
tracker.initialize()

print(f'Detector device: {detector.device}')
print(f'Tracker device: {tracker.device}')
print(f'Experiment: {config.experiment_name}')

if not str(detector.device).startswith('cuda'):
    print('Warning: CUDA not found - running on CPU;')

In [ ]:
from utils import SoccerNetLoader

# Reminder: set dataset.soccernet.password in configs/stage1_baseline.yaml if needed.
if config.dataset is None or config.dataset.soccernet is None:
    raise ValueError('Missing dataset.soccernet configuration in configs/stage1_baseline.yaml')

soccernet_loader = SoccerNetLoader.from_config(config.dataset.soccernet)

try:
    dataset_root = soccernet_loader.ensure_dataset()
except Exception as error:
    raise RuntimeError(
        f'Unable to prepare SoccerNet at {soccernet_loader.root_dir}; set a valid local dataset path in dataset.soccernet.root_dir; reason: {error}'
    ) from error

video_path = soccernet_loader.get_default_video_path()
annotation_payload = soccernet_loader.load_tracking_annotations()
gt_tracks_by_frame = soccernet_loader.map_tracking_annotations_to_tracks(annotation_payload)

print(f'SoccerNet directory: {dataset_root}')
print(f'Video file: {video_path}')
print(f'Frames with GT: {len(gt_tracks_by_frame)}')

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from utils import compute_mot_id_metrics

try:
    import cv2
except Exception:
    cv2 = None
    print('Warning: OpenCV is unavailable - visualization is limited;')

frame_latencies: list[float] = []
pred_tracks_by_frame = {}
preview_frames: list[np.ndarray] = []

max_frames = 120
frame_stride = 2

for frame_index, frame in soccernet_loader.iter_video_frames(
    video_path=video_path,
    max_frames=max_frames,
    stride=frame_stride,
):
    frame_start = time.perf_counter()
    detections = detector.predict(frame)
    tracks = tracker.update(detections=detections, frame_index=frame_index)
    frame_latencies.append(time.perf_counter() - frame_start)
    pred_tracks_by_frame[frame_index] = tracks

    vis_frame = frame.copy()
    if cv2 is not None:
        for track in tracks:
            x1, y1, x2, y2 = [int(value) for value in track.bbox_xyxy]
            cv2.rectangle(vis_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                vis_frame,
                f'ID {track.track_id}',
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

    if len(preview_frames) < 3:
        preview_frames.append(vis_frame)

if not pred_tracks_by_frame:
    raise RuntimeError('No frames were processed;')

preview_count = len(preview_frames)
if preview_count > 0:
    fig, axes = plt.subplots(1, preview_count, figsize=(5 * preview_count, 4))
    if preview_count == 1:
        axes = [axes]
    for index in range(preview_count):
        frame_to_show = preview_frames[index]
        if cv2 is not None:
            frame_to_show = cv2.cvtColor(frame_to_show, cv2.COLOR_BGR2RGB)
        axes[index].imshow(frame_to_show)
        axes[index].set_title(f'Tracking - frame {index + 1}')
        axes[index].axis('off')
    plt.tight_layout()

metric_values = compute_mot_id_metrics(
    ground_truth_by_frame=gt_tracks_by_frame,
    predictions_by_frame=pred_tracks_by_frame,
    iou_threshold=0.5,
)

fps_value = (1.0 / float(np.mean(frame_latencies))) if frame_latencies else 0.0
metrics_state = {
    'MOTA': float(metric_values['MOTA']),
    'IDF1': float(metric_values['IDF1']),
    'ID Swaps': int(metric_values['ID Swaps']),
    'FPS': float(fps_value),
}

print('Status: processing and metric computation completed;')

In [ ]:
import pandas as pd

if 'metrics_state' not in globals():
    raise RuntimeError('metrics_state is missing - run Cell 5 first')

summary_rows = [
    {'Metric': 'MOTA', 'Value': metrics_state['MOTA']},
    {'Metric': 'IDF1', 'Value': metrics_state['IDF1']},
    {'Metric': 'ID Swaps', 'Value': metrics_state['ID Swaps']},
    {'Metric': 'FPS', 'Value': metrics_state['FPS']},
]

metrics_table = pd.DataFrame(summary_rows)
metrics_table['Value'] = metrics_table['Value'].apply(
    lambda value: f'{value:.3f}' if isinstance(value, float) else value
)

print('Metrics: baseline run summary;')
display(metrics_table)